# CALM 

## 0. Setup — Tạo LLM một lần, tất cả agent dùng chung


In [3]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    max_completion_tokens=10000,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

## 1. Planning Agent

In [4]:
from calm.agents.planning_agent import PlanningAgent

agent = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = agent.invoke(query)

print("Approved:", result.get("approved"))
print("Plan:")
for i, step in enumerate(result.get("final_output") or [], 1):
    print(f"  {i}. {step}")

Approved: True
Plan:
  1. {'step_id': 'step-1', 'action': 'retrieve_data', 'agent': 'data_knowledge', 'parameters': {'location': 'Amazon region', 'time_range': {'start': '2023-10-09T00:00:00Z', 'end': '2023-10-16T00:00:00Z'}, 'model': None}, 'expected_output': ['Detailed weather data (temperature, humidity, wind speed)'], 'success_criteria': ['Valid weather data retrieved from NOAA or Global Forecast System in JSON or CSV format with daily updates']}
  2. {'step_id': 'step-2', 'action': 'retrieve_data', 'agent': 'data_knowledge', 'parameters': {'location': 'Amazon region', 'time_range': {'start': None, 'end': None}, 'model': None}, 'expected_output': ['Comprehensive analysis report on fire risk factors'], 'success_criteria': ['Ground-truth data collected and analyzed alongside vegetation moisture, historical fire data, drought conditions, and human activity impacts using satellite imagery']}
  3. {'step_id': 'step-3', 'action': 'retrieve_data', 'agent': 'data_knowledge', 'parameters': 

## 2. RSEN — Xác thực dự đoán

In [6]:
from calm.agents.rsen_module import RSENModule
from calm.memory.chroma_store import ChromaMemoryStore

memory = ChromaMemoryStore(
    collection_name="calm_rsen_memory",
    persist_directory=".chroma",
    k=3,
)
rsen = RSENModule(llm=llm, memory_store=memory, k=3)

result = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={
        "temperature": 35.0,
        "humidity": 0.2,
        "wind_speed": 15,
    },
    spatial_data={
        "fuel_type": "Shrubland",
        "slope": 25,
        "elevation": 500,
    },
)

print("Validation:", result.get("validation_decision"))
print("Rationale:", result.get("final_rationale", "")[:300])

/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13186.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Validation: Plausible
Rationale: The weather report highlights extreme heat and low humidity as primary factors for fire ignition and propagation, corroborated by historical patterns of similar meteorological conditions leading to wildfires. The geospatial analysis reinforces this by indicating that shrubland, combined with a steep


## 3. Wildfire QA Agent

In [7]:
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.qa_agent import WildfireQAAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
memory = ChromaMemoryStore(
    collection_name="calm_qa_memory",
    persist_directory=".chroma",
    k=3,
)

tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None}
data_agent = DataKnowledgeAgent(
    llm=llm, tools=tools, memory_store=memory, config={"dedup_check": True}
)
qa = WildfireQAAgent(
    llm=llm,
    data_agent=data_agent,
    web_search_tool=web,
    memory_store=memory,
    config={"evidence_threshold": 0.65},
)

result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = result.get("final_output") or {}
print("Answer:", out.get("answer", out))
print("Citations:", out.get("citations", []))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7263.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to

Answer: The 2023 Canadian wildfires were primarily caused by a combination of climate change, human activities, and natural events. Key factors include: 
1. **Climate Change**: Scientific consensus indicates that climate change has led to increasingly warmer temperatures and prolonged drought conditions across Canada. This has created an environment that is more conducive to wildfires, as hot and dry conditions can significantly increase the likelihood and intensity of such events.

2. **Human Activities**: Certain land management practices, as well as human-caused ignitions (such as campfires, discarded cigarettes, and other activities), played a role in initiating some of the wildfires. In areas where development encroaches on natural habitats, there can be an increase in human-wildfire interactions.

3. **Natural Events**: Lightning strikes are a significant natural cause of wildfires, particularly in remote areas where human activity is minimal. In 2023, there were numerous instanc

## 4. Full Pipeline — Plan + Evaluator

In [8]:
from calm.agents.evaluator_agent import EvaluatorAgent
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={})
evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})

query = "Assess wildfire risk for California Central Valley next 14 days"
plan_result = planner.invoke(query)
plan = plan_result.get("final_output") or []

print("Plan steps:", len(plan))
eval_result = evaluator.evaluate(
    response={"plan": plan, "query": query},
    query=query,
)

print("Evaluation passed:", eval_result.get("passed"))
print("Average score:", eval_result.get("average_score"))
print("Scores:", eval_result.get("scores", {}))

Plan steps: 5
Evaluation passed: False
Average score: 70.0
Scores: {'data_accuracy': 70, 'explainability': 80, 'jargon_avoidance': 80, 'redundancy_avoidance': 90, 'citation_quality': 60}
